In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import sys
sys.path.append("/home/geraldine/Documents/Research/Projects/BayesGPT/")
print(sys.path)

['/home/a_huangm13/Documents/Research/Projects/BayesGPT', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/bayesgpt', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/tests', '/home/a_huangm13/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/pydev', '/home/a_huangm13/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/jupyter_debug', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python311.zip', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/lib-dynload', '', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/.venv/lib/python3.11/site-packages', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/home/geraldine/Documents/Research/Projects/BayesGPT/', '/home/geraldine/Documents/Research/Projects/BayesGPT/']


In [5]:
import numpy as np

In [24]:
from bayesgpt.simulators.context_manager import ContextManager
from bayesgpt.simulators.model_family import NestedModelFamily
from bayesgpt.simulators.benchmarks.ddms.ddm import DDM
from bayesgpt.simulators.benchmarks.ddms.ddm_priors import ddm_baseline_priors

In [7]:
context_manager = ContextManager()

In [8]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau"]

In [15]:
design_config = context_manager.build_design_config(
    intrinsic_params=intrinsic_params,
    regressed_params=["v", "a"],
    num_regressors=4,
    keep_intercept=True,
    add_interaction=True
)

In [16]:
for k, v in design_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['v', 'a']
u_2 ['v', 'a']
u_3 ['v', 'a']
u_4 ['v', 'a']
u_1:u_2 ['v', 'a']
u_1:u_3 ['v', 'a']
u_1:u_4 ['v', 'a']
u_2:u_3 ['v', 'a']
u_2:u_4 ['v', 'a']
u_3:u_4 ['v', 'a']


In [17]:
random_config = context_manager.build_random_design_config(
    intrinsic_params=intrinsic_params,
    num_regressors=4,
    free_intrinsics=intrinsic_params,
    fixed_intrinsics=[],
    keep_intercept=True,
    free_prob=0.5,
    add_interaction=True
)

In [18]:
for k, v in random_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['tau', 's_v', 's_tau']
u_2 ['a', 'tau', 's_tau']
u_3 ['tau', 's_v']
u_4 ['a', 'tau']
u_1:u_2 ['tau', 's_tau']
u_1:u_3 ['tau', 's_v']
u_1:u_4 ['tau']
u_2:u_3 ['tau']
u_2:u_4 ['a', 'tau']
u_3:u_4 ['tau']


In [19]:
X = context_manager.build_design_matrix(random_config, num_obs=100, keep_intercept=True, max_num_categories=4)

In [20]:
X.shape

(100, 31)

In [22]:
block_width = 3
keys = [k for k in random_config.keys() if k != "1"]

In [23]:
start = {k: i * block_width for i, k in enumerate(keys)}
for k in keys:
    if ":" in k:
        a, b = k.split(":")
        ok = np.allclose(X[:, start[k]], X[:, start[a]] * X[:, start[b]])
        print(k, "OK" if ok else "FAIL")

u_1:u_2 OK
u_1:u_3 OK
u_1:u_4 OK
u_2:u_3 OK
u_2:u_4 OK
u_3:u_4 OK


In [29]:
ddm_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    prior_fun=ddm_baseline_priors(),
    mask_randomizer_kwargs=dict(
        free_intrinsics=["v", "a", "tau"],
        fixed_intrinsics=["s_v", "s_tau"],
        fixed_values={"s_v": 0, "s_tau": 0},
    )
)

In [30]:
sample_kwargs = {
    'min_num_regressors': 1,
    "max_num_regressors": 2,
    "max_num_categories": 2,
    "fixed_config": True
}

samples = ddm_family.batch_sample(
    batch_size=32,
    num_obs=200,
    flatten_param_outputs=True,
    **sample_kwargs
)

In [32]:
for k, v in samples.items():
    if isinstance(v, np.ndarray):
        print(k, v.shape)
    elif isinstance(v, list):
        for i in range(len(v)):
            print(v[i])
    else:
        print(v)

DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
DDM
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['a', 'tau']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['tau'], 'u_2': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['tau'], 'u_2': []}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['tau'], 'u_2': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v'], 'u_2': ['v', 'a', 'tau']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': []}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a', 'tau']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'a'], 'u_2': ['a']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'tau'], 'u_2': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': ['v', 'tau'], 'u_2': ['v']}
{'1': ['v', 'a', 'tau', 's_v', 's_tau'], 'u_1': []}
{'1': ['v', 